Compare 04.1 (Zero-Shot NLI) vs 04.2 (GoEmotions) Classifiers
===============================================================
Both notebooks classify the same 1,105 songs from `03_lyrics_trans.csv`, but
with different label taxonomies:

- **04.1 zero-shot** (`bart-large-mnli`): 10 custom song-emotion labels
  (`love`, `longing`, `joy`, `heartbreak`, `grief`, `despair`, `hope`,
  `lonely`, `sensual`, `anger`), competing scores (softmax, sum to ~1).
- **04.2 GoEmotions** (`roberta-base-go_emotions-onnx`): 28 fixed labels,
  independent sigmoid scores (not competing, can all score high at once).

The two taxonomies only overlap on 4 labels: `love`, `joy`, `grief`, `anger`.
This notebook compares coverage, score correlation on the shared labels, and
dominant-emotion agreement — it does not try to reconcile the full label sets,
since most of each taxonomy has no equivalent in the other.

In [1]:
import pandas as pd
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent

zeroshot = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "04.1_emotion_scores_zeroshot.csv")
goemotions = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "04.2_emotion_scores_goemotions.csv")

SHARED_LABELS = ["love", "joy", "grief", "anger"]

assert set(zeroshot.spotify_uri) == set(goemotions.spotify_uri), "Expected identical track sets"
print(f"Comparing {len(zeroshot)} songs across both classifiers.")
print(f"Shared labels (present in both taxonomies): {SHARED_LABELS}")

Comparing 1105 songs across both classifiers.
Shared labels (present in both taxonomies): ['love', 'joy', 'grief', 'anger']


### 1. Coverage — how many songs each classifier could confidently score

In [1]:
merged = zeroshot.merge(
    goemotions[["spotify_uri", "dominant_emotion"] + [f"emotion_{e}" for e in SHARED_LABELS]],
    on="spotify_uri",
    suffixes=("_zeroshot", "_goemotions"),
)

n = len(merged)
zs_unclassified = (merged.dominant_emotion_zeroshot == "unclassified").sum()
ge_unclassified = (merged.dominant_emotion_goemotions == "unclassified").sum()
both_unclassified = (
    (merged.dominant_emotion_zeroshot == "unclassified")
    & (merged.dominant_emotion_goemotions == "unclassified")
).sum()

coverage = pd.DataFrame({
    "metric": ["unclassified", "classified"],
    "zeroshot (04.1)": [zs_unclassified, n - zs_unclassified],
    "goemotions (04.2)": [ge_unclassified, n - ge_unclassified],
})
coverage["zeroshot (04.1) %"] = (coverage["zeroshot (04.1)"] / n * 100).round(1)
coverage["goemotions (04.2) %"] = (coverage["goemotions (04.2)"] / n * 100).round(1)

print(f"Total songs: {n}")
print(f"Unclassified in both: {both_unclassified}")
display(coverage)

Total songs: 1105
Unclassified in both: 107

      metric  zeroshot (04.1)  goemotions (04.2)  zeroshot (04.1) %  goemotions (04.2) %
unclassified              108                201                9.8                 18.2
  classified              997                904               90.2                 81.8


GoEmotions leaves more songs unclassified (`UNCLASSIFIED_THRESHOLD = 0.30` in
04.2) — expected, since independent sigmoid scores rarely all land near 1 the
way a forced-competition softmax does in 04.1, so fewer songs clear a fixed bar.

### 2. Dominant emotion distribution — each classifier's own taxonomy

In [1]:
print("=== Zero-shot (04.1) dominant emotion distribution ===")
display(zeroshot.dominant_emotion.value_counts())

print("\n=== GoEmotions (04.2) dominant emotion distribution ===")
display(goemotions.dominant_emotion.value_counts())

=== Zero-shot (04.1) dominant emotion distribution ===
dominant_emotion
sensual         366
longing         209
lonely          118
unclassified    108
heartbreak      101
love             94
anger            67
hope             23
despair          10
grief             5
joy               4

=== GoEmotions (04.2) dominant emotion distribution ===
dominant_emotion
neutral           312
love              233
unclassified      201
sadness            83
desire             47
amusement          46
admiration         28
disappointment     24
curiosity          23
annoyance          18
confusion          12
joy                11
fear               10
caring              9
remorse             8
disapproval         7
approval            7
optimism            7
gratitude           6
anger               4
excitement          4
realization         2
nervousness         1
disgust             1
embarrassment       1


### 3. Score correlation on the 4 shared labels

For labels that exist in both taxonomies, do the two models' *scores* agree
directionally across all 1,105 songs — regardless of which label either model
picked as dominant?

In [1]:
correlations = {
    e: merged[f"emotion_{e}_zeroshot"].corr(merged[f"emotion_{e}_goemotions"])
    for e in SHARED_LABELS
}
corr_df = pd.DataFrame.from_dict(correlations, orient="index", columns=["pearson_r"]).round(3)
display(corr_df)

       pearson_r
love       0.483
joy        0.284
grief      0.387
anger      0.295


Moderate positive correlation on all 4 shared labels is expected, not perfect
agreement — the two models score under structurally different assumptions
(competing softmax vs. independent sigmoid), so absolute scores aren't
directly comparable even for the same label name. Correlation captures whether
they tend to agree on *which songs* score relatively higher/lower on a label,
not whether the magnitudes match.

### 4. Dominant-emotion agreement, restricted to the shared label set

Of the songs where GoEmotions' dominant emotion happens to be one of the 4
shared labels (and zero-shot did produce a classification), how often does
zero-shot's dominant emotion agree exactly?

In [1]:
subset = merged[
    merged.dominant_emotion_goemotions.isin(SHARED_LABELS)
    & (merged.dominant_emotion_zeroshot != "unclassified")
]

agreement_rate = (subset.dominant_emotion_zeroshot == subset.dominant_emotion_goemotions).mean()

print(f"Songs where GoEmotions dominant \u2208 shared labels: {len(subset)}")
print(f"Exact agreement rate with zero-shot dominant: {agreement_rate:.1%}")
print("\nBreakdown of GoEmotions dominant label within this subset:")
display(subset.dominant_emotion_goemotions.value_counts())

Songs where GoEmotions dominant ∈ shared labels: 248
Exact agreement rate with zero-shot dominant: 23.4%

Breakdown of GoEmotions dominant label within this subset:
dominant_emotion_goemotions
love     233
joy       11
anger      4


Low exact agreement is expected here too: 04.1's `sensual` and `longing`
labels (which don't exist in GoEmotions) absorb a large share of zero-shot's
dominant picks (see distribution above), so even when GoEmotions confidently
says `love`, zero-shot often prefers a related-but-distinct label from its
richer romantic-emotion vocabulary rather than `love` itself. This is a
taxonomy-granularity mismatch, not necessarily a disagreement about content.

### 5. Regional comparison on shared labels

In [1]:
titles = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "00_titles.csv")[["spotify_uri", "region"]]
shared_cols = [f"emotion_{e}" for e in SHARED_LABELS]

zs_region = titles.merge(zeroshot[["spotify_uri"] + shared_cols], on="spotify_uri", how="inner")
ge_region = titles.merge(goemotions[["spotify_uri"] + shared_cols], on="spotify_uri", how="inner")

zs_summary = zs_region.groupby("region")[shared_cols].mean().round(3)
ge_summary = ge_region.groupby("region")[shared_cols].mean().round(3)

zs_summary.columns = [f"{c.replace('emotion_', '')}_zeroshot" for c in zs_summary.columns]
ge_summary.columns = [f"{c.replace('emotion_', '')}_goemotions" for c in ge_summary.columns]

regional_comparison = zs_summary.join(ge_summary)
# Interleave columns per label so zeroshot/goemotions sit side by side
ordered_cols = [c for e in SHARED_LABELS for c in (f"{e}_zeroshot", f"{e}_goemotions")]
regional_comparison = regional_comparison[ordered_cols]

display(regional_comparison)

output_path = PROJECT_ROOT / "data" / "processed" / "07_classifier_comparison_regional.csv"
regional_comparison.to_csv(output_path)
print(f"Saved to {output_path}")

           love_zeroshot  love_goemotions  joy_zeroshot  joy_goemotions  grief_zeroshot  grief_goemotions  anger_zeroshot  anger_goemotions
region                                                                                                                                     
Argentina          0.565            0.195         0.206           0.024           0.496             0.003           0.466             0.018
Colombia           0.611            0.261         0.257           0.039           0.475             0.003           0.492             0.019
Global             0.679            0.215         0.298           0.030           0.576             0.003           0.494             0.025
Japan              0.634            0.272         0.321           0.052           0.417             0.005           0.273             0.016
Singapore          0.655            0.182         0.299           0.024           0.530             0.003           0.442             0.021
Spain              0

### Summary

- **Coverage**: zero-shot classifies more songs (fewer `unclassified`) than
  GoEmotions at its current threshold, since forced-competition softmax scores
  concentrate probability mass more than independent sigmoids do.
- **Score correlation** on the 4 shared labels is positive but moderate —
  the models don't disagree about direction, but their scoring mechanics
  aren't directly comparable in magnitude.
- **Dominant-emotion agreement** is low mainly because 04.1's richer
  romantic/melancholic vocabulary (`sensual`, `longing`, `heartbreak`,
  `lonely`) captures nuance that GoEmotions collapses into `love` or
  `sadness` — a taxonomy resolution difference, not necessarily a
  disagreement about what the song is about.
- Neither classifier is strictly "more correct" here — 04.1 was designed for
  song-emotion nuance, 04.2 for speed via a fixed general-purpose taxonomy.
  Which one is more useful depends on whether the downstream analysis needs
  song-specific emotional vocabulary or just fast, general sentiment.